In [23]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold




# EXC 1

In [15]:
np.random.seed(42)

n = 100
X = np.random.normal(12, 1, n)
epsilon = np.random.normal(0, 1, n)
Y = 2 * X + epsilon

df = pd.DataFrame({
    "X": X,
    "Y": Y
})

df.head()

,X,Y
0,12.496714,23.578058
1,11.861736,23.302826
2,12.647689,24.952663
3,13.523030,26.243782
4,11.765847,23.370408


In [11]:
model = LinearRegression()
model.fit(df[["X"]], df["Y"])


LinearRegression()

In [ ]:
print(model.intercept_)
print(model.coef_)


np.float64(1.7265137531212744)

In [17]:
y_hat = model.predict(df[["X"]])
print("predicted y")
y_hat[:5]

predicted y


array([24.92969828, 23.75070658, 25.21001889, 26.83530261, 23.57266523])

In [18]:
residuals = df["Y"] - y_hat
residuals

0    -1.351641
1    -0.447881
2    -0.257356
3    -0.591520
4    -0.202258
        ...   
95    0.168231
96   -0.848864
97    0.183695
98    0.051513
99   -1.184004
Name: Y, Length: 100, dtype: float64

In [21]:
mse = sum(residuals**2)/len(residuals)
mse

0.8836324822912887

In [24]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [29]:
mse_list= []
for train_index, test_index in kf.split(df):
    
    X_train = df.iloc[train_index][["X"]]
    y_train = df.iloc[train_index]["Y"]

    X_test = df.iloc[test_index][["X"]]
    y_test = df.iloc[test_index]["Y"]

    model = LinearRegression()
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    mse = np.mean((y_test - y_pred)**2)

    mse_list.append(mse)

cv_mse = np.mean(mse_list)

cv_mse

np.float64(0.9627676030856614)

# EXC 2

In [31]:
np.random.seed(42)

n = 100
p = 10

X = np.random.normal(0, 1, (n, p))
epsilon = np.random.normal(0, 1, n)

Y = 3*X[:, 0] + 2*X[:, 1] - 1*X[:, 2] + epsilon

columns = [f"X{i}" for i in range(1, 11)]
df = pd.DataFrame(X, columns=columns)
df["Y"] = Y

In [33]:
df.head(3)

,X1,X2,X3,X4,X5,X6,X7,X8,X9,X10,Y
0,0.496714,-0.138264,0.647689,1.523030,-0.234153,-0.234137,1.579213,0.767435,-0.469474,0.542560,1.965281
1,-0.463418,-0.465730,0.241962,-1.913280,-1.724918,-0.562288,-1.012831,0.314247,-0.908024,-1.412304,-1.639041
2,1.465649,-0.225776,0.067528,-1.424748,-0.544383,0.110923,-1.150994,0.375698,-0.600639,-0.291694,3.937496


In [34]:
full_predictors = [f"X{i}" for i in range(1, 11)]
reduced_predictors = ["X1", "X2", "X3"]

In [35]:
kf = KFold(n_splits=10, shuffle=True, random_state=42)

full_mse_list = []
reduced_mse_list = []

for train_index, test_index in kf.split(df):
    train_df = df.iloc[train_index]
    test_df = df.iloc[test_index]

    # Full model
    model_full = LinearRegression()
    model_full.fit(train_df[full_predictors], train_df["Y"])
    y_pred_full = model_full.predict(test_df[full_predictors])
    full_mse = np.mean(test_df["Y"] - y_pred_full)**2
    full_mse_list.append(full_mse)

    # Reduced model
    model_reduced = LinearRegression()
    model_reduced.fit(train_df[reduced_predictors], train_df["Y"])
    y_pred_reduced = model_reduced.predict(test_df[reduced_predictors])
    reduced_mse = np.mean(test_df["Y"] - y_pred_reduced)**2
    reduced_mse_list.append(reduced_mse)

cv_mse_full = np.mean(full_mse_list)
cv_mse_reduced = np.mean(reduced_mse_list)

print("Full model CV MSE:", cv_mse_full)
print("Reduced model CV MSE:", cv_mse_reduced)

Full model CV MSE: 0.17403616729332333
Reduced model CV MSE: 0.14741386087868538


# EXC 3

In [75]:
np.random.seed(37)
n = 100
X = np.random.normal(0, 5, n)
epsilon = np.random.normal(0, 1, n)
Y = 2 * X + epsilon

df = pd.DataFrame({
    "X": X,
    "Y": Y
})

In [76]:
model = LinearRegression()
model.fit(df[["X"]], df["Y"])
print(model.intercept_)
print(model.coef_)

-0.020213115152234784
[2.01542647]


In [77]:
df

,X,Y
0,-0.272318,0.307505
1,3.371540,7.841666
2,1.733235,3.177231
3,-6.501731,-10.792307
4,7.592559,14.617509
...,...,...
95,-0.156968,1.505193
96,-2.171106,-3.460182
97,1.998698,3.734704
98,5.659400,11.145750


In [78]:
y_hat = model.predict(df[["X"]])
y_hat[:5]

array([ -0.56905015,   6.77487852,   3.47299484, -13.12397358,
        15.28203207])

In [79]:
apparent_error = np.mean((df["Y"] - y_hat)**2)
apparent_error

np.float64(0.9657056770091619)

In [80]:
print("optimism estimate")
sigma2 = 1
d = 2
n = 100

optimism = (2 * sigma2 * d) / n
optimism

optimism estimate


0.04

In [81]:
print("mallows' C estimate of prediction error")
corrected_error = apparent_error + optimism
corrected_error

mallows' C estimate of prediction error


np.float64(1.005705677009162)

In [82]:
kf = KFold(n_splits=10, shuffle=True, random_state=42)

cv_mse = []

for train_index, test_index in kf.split(df):

    train_df = df.iloc[train_index]
    test_df = df.iloc[test_index]

    model = LinearRegression()
    model.fit(train_df[["X"]], train_df["Y"])

    y_pred = model.predict(test_df[["X"]])

    mse = np.mean((test_df["Y"] - y_pred)**2)

    cv_mse.append(mse)

cv_error = np.mean(cv_mse)

In [83]:
print(apparent_error, corrected_error, cv_error)

0.9657056770091619 1.005705677009162 1.0080982312762563
